# Documents that learn

*Read structure out of a contract, attach it to the document it came from, and query it with SQL++ — no migration, no second store.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/data-model/01_documents_that_learn.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** A document can gain structured fields after it is stored, with no schema change, and become queryable as data without ceasing to be prose.
**Result.** Twenty contracts arrive as walls of text and end up answering `SELECT ... WHERE governing_law = "California" AND agreement_date > "2015"`.
**Requires.** couchbase · llm
**Read** ~8 min · **Run** ~3 min · **Cost** a few cents

Extracting fields from a document with a language model is not the interesting part of this
notebook. You know it works and there are a hundred posts about the prompt.

The interesting part is **what happens to the document next**. In a relational stack, new
fields mean a migration: a table to design, a key to join on, a backfill to run, and from then
on two places that can disagree about the same contract. Here the fields go onto the document
they were read from, and the query language finds them immediately.

**What this notebook does**

1. Stores twenty contracts exactly as they arrive — a title and a wall of text.
2. Asks a model for the handful of terms a lawyer would look for.
3. Attaches them two ways — **onto** the document, and **beside** it as its own record — and
   says when each is right.
4. Queries both with SQL++, as though they had been columns all along.

That is the whole arc, and it stops there on purpose. Vectors, chunking, and search over the
enriched documents are [`data-model/02`](02_adding_retrieval.ipynb) — this one is about the
document.

In [ ]:
# --- Setup. Works in a local checkout and on Colab. -------------------------
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["couchbase", "llm"])

## 1. What arrives

Twenty real commercial contracts, filed with the SEC and annotated by lawyers as part of
[CUAD](https://www.atticusprojectai.org/cuad). Right now none of that annotation is used —
they are just documents, the way a document looks when it lands in a system: a filename and
thirty thousand characters of prose.

Store them as they are. Enriching something before you store it is a pipeline; enriching
something already stored is what this notebook is about.

In [2]:
from cbnb.datasets import load_cuad

contracts = load_cuad().contracts
print(f"{len(contracts)} contracts, {contracts.text.str.len().median():,.0f} characters each (median)")
print()
print(contracts.text.iloc[0][:260].strip(), "...")

20 contracts, 29,316 characters each (median)

Exhibit 10.23 Corporate Address Fannin South Professional Building, Suite 140 7707 Fannin Street Houston, Texas 77054 t: 832.968.4888

CONSULTING AGREEMENT

July 20, 2018

Gianluca Rotino

Dear Gianluca:

Kiromic, Inc, a Delaware corporation (the "Company"), i ...


In [3]:
from cbnb.couchbase_io import connect, ensure_collection, upsert_docs

BUCKET, SCOPE = settings.cb_bucket, "contract_intelligence"
COLLECTION, TERMS = "contracts", "contract_terms"

cluster = connect(settings)
documents = ensure_collection(cluster, BUCKET, SCOPE, COLLECTION)

upsert_docs(documents, {
    f"contract::{row.contract_id}": {
        "type": COLLECTION,
        "contract_id": int(row.contract_id),
        "source_title": row.title,
        "text": row.text,
    }
    for row in contracts.itertuples(index=False)
}, progress=False)

stored = documents.get("contract::0").content_as[dict]
print("fields on the stored document:", list(stored))
print(f"text: {len(stored['text']):,} characters")

fields on the stored document: ['type', 'contract_id', 'source_title', 'text']
text: 18,403 characters


## 2. Read it the way a lawyer would

The well-understood step: ask a model for the facts somebody would otherwise pay a junior
associate to find, with a schema so the answer comes back as data rather than a paragraph.

Two practical notes that matter more than the prompt wording:

**Send the whole contract.** An earlier draft of this notebook truncated at 14,000 characters
and `governing_law` came back empty for several contracts — not because the model failed, but
because the clause was at character 30,000. Truncation raises nothing; it quietly converts a
field that exists into a field that does not.

**Let fields be empty.** Some contracts genuinely have no governing-law clause. A schema that
demands a value gets you a confident invention, which is worse than a blank — the failure
[`flows/01`](../flows/01_rag_that_you_can_trust.ipynb) spends a whole notebook measuring.

In [4]:
from pydantic import BaseModel, Field

from cbnb.llm import LLM


class Terms(BaseModel):
    """The handful of facts that make a contract findable."""

    title: str = Field(description="the contract's own title, e.g. 'DISTRIBUTOR AGREEMENT'")
    parties: list[str] = Field(description="the legal entities that signed it")
    governing_law: str = Field(description="the jurisdiction whose law governs, e.g. 'Texas'; empty if not stated")
    agreement_date: str = Field(description="the date signed, as YYYY-MM-DD; empty if not stated")
    term_years: float | None = Field(description="initial term length in years if stated as a fixed period, else null")
    auto_renews: bool = Field(description="true if the term extends automatically unless notice is given")


SYSTEM = ("You read commercial contracts and extract their key terms. Use only what the text "
          "says. Leave a field empty or null rather than guessing.")

llm = LLM()


def extract(row):
    return llm.structured(f"Contract:\n\n{row.text}", Terms, system=SYSTEM, max_tokens=500)


# Twenty contracts, six at a time. Cached, so re-running this cell is free.
terms = llm.map(list(contracts.itertuples(index=False)), extract, max_workers=6)

  20/20

In [5]:
import pandas as pd

pd.DataFrame([
    {"id": row.contract_id, "title": t.title[:34], "governing_law": t.governing_law,
     "agreement_date": t.agreement_date, "term_years": t.term_years,
     "auto_renews": t.auto_renews}
    for row, t in zip(contracts.itertuples(index=False), terms)
]).head(8)

,id,title,governing_law,agreement_date,term_years,auto_renews
0,0,CONSULTING AGREEMENT,Texas,2018-07-20,NaN,False
1,1,COOPERATION AGREEMENT (2014 Amendm,People's Republic of China,2014-01-24,NaN,False
2,2,JOINT CONTENT LICENSE AGREEMENT,California,2018-02-01,3.0,True
3,3,DISTRIBUTORSHIP AGREEMENT,Ohio,2018-03-29,1.0,True
4,4,SOFTWARE DEVELOPMENT AGREEMENT,Washington,2018-12-03,NaN,False
5,5,SOFTWARE LICENSE AND MAINTENANCE A,Nova Scotia,2000-05-01,NaN,False
6,6,TRANSPORTATION SERVICES AGREEMENT,Texas,2003-12-23,3.0,True
7,7,COLLABORATION AGREEMENT,Delaware,2020-04-14,NaN,False


## 3. Embed it, or reference it?

This is the step a relational stack turns into a project, and the step where a document store
hands you a decision worth making on purpose. It has a standard name — **embedding versus
referencing** — and both halves are one write, with no migration either way.

**Embed** — the terms become fields on the contract document itself. One document, one atomic
write, and every read sees the text and the terms together.

**Reference** — the terms become their own document, keyed from the contract:
`contract::0` keeps its text, `contract::0::terms` holds what was extracted. The source is never
touched. When the separate document holds attributes *derived* from the source, it is usually
called a **derived document**; Data Vault modelling calls the same shape a *satellite*.

They fail differently, which is the point, so this notebook does both.

### Embedding: the terms go on the contract

No `ALTER TABLE`. No `contract_terms` table, no foreign key, no backfill, no window during
which half the rows have the new column. The fields go onto the document.

And writing them does not rewrite the document. **Subdocument mutation** sends only the paths
being changed — a few hundred bytes — and the server applies them in place. The thirty thousand
characters of contract text are not read, not sent, and not rewritten.

In [6]:
import couchbase.subdocument as SD


def as_rfc3339(value):
    """A date the way Couchbase indexes it. Unparseable input becomes None, not today."""
    parsed = pd.to_datetime(value, errors="coerce")
    return None if pd.isna(parsed) else parsed.strftime("%Y-%m-%dT%H:%M:%SZ")


for row, t in zip(contracts.itertuples(index=False), terms):
    fields = {
        "title": t.title,
        "parties": t.parties,
        "governing_law": t.governing_law,
        "agreement_date": as_rfc3339(t.agreement_date),
        "term_years": t.term_years,
        "auto_renews": t.auto_renews,
        # A marker, not decoration: anything that enriches on write needs to know
        # what it has already done. See "Where to take this".
        "enriched_at": pd.Timestamp.now("UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    documents.mutate_in(
        f"contract::{row.contract_id}",
        # Only paths that have a value. An absent clause stays absent rather than
        # being stored as an explicit null.
        [SD.upsert(name, value) for name, value in fields.items() if value not in (None, "")],
    )

print(f"enriched {len(contracts)} documents in place")

enriched 20 documents in place


In [7]:
after = documents.get("contract::0").content_as[dict]
print("fields now:", sorted(after), "\n")
for key in ("title", "parties", "governing_law", "agreement_date", "term_years", "auto_renews"):
    print(f"  {key:16} {after.get(key, '(absent)')}")
print(f"\ntext still {len(after['text']):,} characters, untouched")

fields now: ['agreement_date', 'auto_renews', 'contract_id', 'enriched_at', 'governing_law', 'parties', 'source_title', 'text', 'title', 'type'] 

  title            CONSULTING AGREEMENT
  parties          ['Kiromic, Inc.', 'Gianluca Rotino']
  governing_law    Texas
  agreement_date   2018-07-20T00:00:00Z
  term_years       (absent)
  auto_renews      False

text still 18,403 characters, untouched


### Absent stays absent

A contract with no governing-law clause has no `governing_law` key. Not an empty string, not a
null — the key is not there.

In a table that column would be `NULL` on every row that lacked it, `NOT NULL` would have been
an argument someone lost, and every query would carry a clause to cope. Here the shape of each
document describes that document, and `IS NOT MISSING` is the difference between "we did not
find one" and "there is not one".

In [8]:
for i, t in enumerate(terms):
    if not t.governing_law:
        doc = documents.get(f"contract::{i}").content_as[dict]
        print(f"contract::{i} — keys: {sorted(k for k in doc if k != 'text')}")
        print(f"  'governing_law' in document: {'governing_law' in doc}")

contract::10 — keys: ['auto_renews', 'contract_id', 'enriched_at', 'parties', 'source_title', 'title', 'type']
  'governing_law' in document: False
contract::12 — keys: ['auto_renews', 'contract_id', 'enriched_at', 'parties', 'source_title', 'title', 'type']
  'governing_law' in document: False


### Referencing: the terms get their own document

The same extraction, written as a separate document. The key *is* the relationship:
`contract::0::terms` belongs to `contract::0` because of how it is named, not because of a
foreign key anybody declared.

Nothing about the source changes — no mutation, no new revision, no chance of two enrichers
colliding on one document.

It goes in **its own collection**. Putting derived documents in with their sources means every
query needs a `WHERE type = ...` discriminator, and the first version of this notebook got that
wrong: `COUNT(*)` returned 40 for 20 contracts, and the governing-law tally was exactly double.
Collections are cheap; a discriminator you must remember on every query is not.

In [9]:
terms_records = {}
for row, t in zip(contracts.itertuples(index=False), terms):
    record = {
        "type": TERMS,
        "contract_id": int(row.contract_id),
        "of": f"contract::{row.contract_id}",          # what this describes
        "extractor": llm.model,                         # and what produced it
        "extracted_at": pd.Timestamp.now("UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
        **{k: v for k, v in {
            "title": t.title,
            "parties": t.parties,
            "governing_law": t.governing_law,
            "agreement_date": as_rfc3339(t.agreement_date),
            "term_years": t.term_years,
            "auto_renews": t.auto_renews,
        }.items() if v not in (None, "")},
    }
    terms_records[f"contract::{row.contract_id}::terms"] = record

term_docs = ensure_collection(cluster, BUCKET, SCOPE, TERMS)
upsert_docs(term_docs, terms_records, progress=False)
print(f"wrote {len(terms_records)} derived documents; sources untouched")
print()
print(term_docs.get("contract::2::terms").content_as[dict])

wrote 20 derived documents; sources untouched

{'type': 'contract_terms', 'contract_id': 2, 'of': 'contract::2', 'extractor': 'z-ai/glm-5.3', 'extracted_at': '2026-09-17T01:01:15Z', 'title': 'JOINT CONTENT LICENSE AGREEMENT', 'parties': ['WPT Enterprises, Inc.', 'Zynga Inc.', 'Zynga Game Ireland Limited'], 'governing_law': 'California', 'agreement_date': '2018-02-01T00:00:00Z', 'term_years': 3.0, 'auto_renews': True}


### Choosing between them

| | on the document | beside it |
| --- | --- | --- |
| one read gets everything | yes | needs a second fetch or a join |
| two enrichers running at once | they contend on one document | independent, one record each |
| keeping the source pristine | it gains fields | never touched |
| several versions of an extraction | last write wins | `::terms::v2` sits alongside |
| what produced this value | needs its own fields | naturally on the record |

**Mutate when the enrichment is intrinsic and settled** — a handful of stable facts that belong
to the contract and that one process owns.

**Decorate when the enrichment is an opinion** — produced by a particular model at a particular
time, likely to be redone, possibly disagreed with by a second extractor. The record carries its
own provenance, and superseding it is a write rather than a migration.

The second pattern is also what the next notebook is already doing without saying so: **chunks
are decoration.** A contract split into thirty chunks is thirty records that exist because of
the source and are keyed from it, and nobody would suggest storing them inside the contract.

## 4. The prose is now data

Nothing above declared a schema, and SQL++ does not need one — but it does want an index, for
the same reason any database does.

In [10]:
for statement in [
    f"CREATE PRIMARY INDEX IF NOT EXISTS ON `{BUCKET}`.`{SCOPE}`.`{COLLECTION}`",
    f"CREATE INDEX IF NOT EXISTS idx_law_date ON `{BUCKET}`.`{SCOPE}`.`{COLLECTION}`"
    f"(governing_law, agreement_date)",
]:
    list(cluster.query(statement))
print("indexes ready")

indexes ready


In [11]:
rows = cluster.query(f"""
    SELECT governing_law, COUNT(*) AS contracts
    FROM `{BUCKET}`.`{SCOPE}`.`{COLLECTION}`
    WHERE governing_law IS NOT MISSING
    GROUP BY governing_law
    ORDER BY contracts DESC
""")
pd.DataFrame(list(rows))

,contracts,governing_law
0,4,New York
1,3,California
2,2,Texas
3,1,Massachusetts
4,1,Nova Scotia
5,1,Ohio
6,1,People's Republic of China
7,1,Virginia
8,1,Washington
9,1,Arizona


A distribution of governing law across a contract portfolio, from documents that were prose
twenty minutes ago and were never given a schema.

The same question against the *sibling* records needs a join — and because the relationship is
the key, it is a join by key. No index, no scan: SQL++ is told exactly which document to fetch.

In [12]:
rows = cluster.query(f"""
    SELECT c.source_title, t.governing_law, t.extractor
    FROM `{BUCKET}`.`{SCOPE}`.`{COLLECTION}` AS c
    JOIN `{BUCKET}`.`{SCOPE}`.`{TERMS}` AS t
      ON KEYS "contract::" || TOSTRING(c.contract_id) || "::terms"
    WHERE t.governing_law IS NOT MISSING
    ORDER BY t.governing_law
    LIMIT 6
""")
pd.DataFrame(list(rows))

,extractor,governing_law,source_title
0,z-ai/glm-5.3,Arizona,"HALITRON,INC_03_01_2005-EX-10.15-SPONSORSHIP A..."
1,z-ai/glm-5.3,California,"SightLife Surgical, Inc. - STRATEGIC SALES _ M..."
2,z-ai/glm-5.3,California,"NETGEAR,INC_04_21_2003-EX-10.16-DISTRIBUTOR AG..."
3,z-ai/glm-5.3,California,AlliedEsportsEntertainmentInc_20190815_8-K_EX-...
4,z-ai/glm-5.3,Delaware,ANIXABIOSCIENCESINC_06_09_2020-EX-10.1-COLLABO...
5,z-ai/glm-5.3,Israel,InmodeLtd_20190729_F-1A_EX-10.9_11743243_EX-10...


Now the question this notebook exists to answer — one that needs *two* extracted fields at
once, neither of which was a field when the contracts were stored.



In [13]:
rows = cluster.query(f"""
    SELECT title, governing_law, agreement_date, term_years, auto_renews
    FROM `{BUCKET}`.`{SCOPE}`.`{COLLECTION}`
    WHERE governing_law = "California"
      AND agreement_date > "2015"
    ORDER BY agreement_date
""")
pd.DataFrame(list(rows))

,agreement_date,auto_renews,governing_law,term_years,title
0,2017-04-28T00:00:00Z,False,California,3,STRATEGIC SALES & MARKETING AGREEMENT
1,2018-02-01T00:00:00Z,True,California,3,JOINT CONTENT LICENSE AGREEMENT


In [14]:
rows = cluster.query(f"""
    SELECT COUNT(*) AS n,
           COUNTN(term_years) AS with_a_fixed_term,
           SUM(CASE WHEN auto_renews THEN 1 ELSE 0 END) AS auto_renewing
    FROM `{BUCKET}`.`{SCOPE}`.`{COLLECTION}`
""")
pd.DataFrame(list(rows))

,auto_renewing,n,with_a_fixed_term
0,10,20,10


## What this bought

**One write, not two.** The terms live on the document they were read from. There is no second
store holding contract metadata, so there is no window in which the two disagree and no sync
job whose failure you discover later.

**No migration.** The documents gained six fields after they were stored. Nothing was altered,
nothing was backfilled, and contracts without a clause simply lack that key.

**Prose and data at the same time.** The `text` field is byte-identical to what was stored in
section 1. The document did not stop being a document when it became queryable.

## Where to take this

- **Add retrieval.** These documents are still only searchable by their structured fields.
  [`data-model/02`](02_adding_retrieval.ipynb) adds chunks — decoration records, by the pattern
  above — and the modelling decision that filtering a vector search forces on you.
- **Measure the extraction.** CUAD ships lawyer-written spans for these same fields, so
  "is `governing_law` right?" is checkable rather than a vibe. Nobody should ship an enrichment
  pipeline they have not scored, and the apparatus is already in
  [`flows/01`](../flows/01_rag_that_you_can_trust.ipynb).
- **Make it happen on write.** Everything here was triggered by running a cell. In production
  the trigger is the write itself, and there are two honest paths.

  **Couchbase Eventing** runs a JavaScript handler on every mutation and can call out through a
  cURL binding, so "document arrives → model reads it → fields go back on" is expressible
  inside the database. Two caveats decide whether you want that. A handler must finish before
  its timeout and a slow one builds a backlog, so a multi-second model call belongs in a
  **Timer callback** that `OnUpdate` enqueues rather than in `OnUpdate` itself. And a handler
  that writes back to the collection it watches will **re-trigger itself** — which is what
  `enriched_at` is for: a marker the handler checks before doing any work. Note that Eventing
  is a paid-tier service on Capella, so unlike the rest of this notebook you cannot try it on
  the free tier.

  **A change-feed consumer outside the database** is less elegant and often the better answer
  for this particular job. Model calls are slow, cost money, fail in ways that want retry with
  backoff, and get cheaper in batches — all things easier to control in your own process than
  inside a mutation handler.

- **Re-extract with a better prompt.** `enriched_at` is on every document, so a second pass is
  an update rather than a migration, and the interesting question becomes which documents
  *changed* — a diff, not a rebuild.

> **On Couchbase AI Data Plane** — the extract-and-write-back loop above is yours to run and
> yours to re-run when a document changes. The managed service is aimed squarely at closing
> that gap, so enrichment follows the document rather than following a cell you remembered to
> execute. What does not change is the modelling: the fields still belong on the document, and
> deciding which fields are worth extracting is still a judgement about your data.

In [15]:
# Everything this notebook created, in one scope. `data-model/02` builds on it,
# so leave it in place if you plan to run that next.
# from cbnb.couchbase_io import drop_demo_data
# drop_demo_data(cluster, BUCKET, SCOPE)